# Eval Results Aggregation

This notebook aggregates `eval/results.json` into Table II-ready summaries for the current M-Core run.

In [9]:
from pathlib import Path
import json
import re
import pandas as pd

In [10]:
RESULTS_PATH = (Path.cwd() / "results.json").resolve()
rows = json.loads(RESULTS_PATH.read_text(encoding='utf-8'))
df = pd.DataFrame(rows)
df.head()

,id,archetype,task_prompt,valid,grounded,params_valid,struct_comp,valid_issues,grounded_issues,params_issues,struct_issues,struct_checks,bt_path,trace_comp,exec_status,runtime_ms,tick_count,feasible,goal_satisfied,success
0,task01_hold_bread,unlabeled,"Go to pantry_storage, detect bread, and pick i...",True,True,True,True,[],[],[],[],"{'actions_ok': True, 'locations_ok': True, 'me...",/Users/jonathansalfity/Documents/dev/pyrobosim...,None,SUCCESS,10046,11,True,True,True
1,task02_hold_snacks,unlabeled,"Navigate to pantry_storage, detect snacks, and...",True,True,True,True,[],[],[],[],"{'actions_ok': True, 'locations_ok': True, 'me...",/Users/jonathansalfity/Documents/dev/pyrobosim...,None,SUCCESS,10047,11,True,True,True
2,task03_hold_soda,unlabeled,"Go to fridge_storage, detect soda, and pick it...",True,True,True,True,[],[],[],[],"{'actions_ok': True, 'locations_ok': True, 'me...",/Users/jonathansalfity/Documents/dev/pyrobosim...,None,SUCCESS,10049,11,True,True,True
3,task04_hold_butter,unlabeled,"Navigate to fridge_storage, detect butter, and...",True,True,True,True,[],[],[],[],"{'actions_ok': True, 'locations_ok': True, 'me...",/Users/jonathansalfity/Documents/dev/pyrobosim...,None,SUCCESS,10052,11,True,True,True
4,task05_hold_waste_desk,unlabeled,"Go to desk_desktop, detect waste, and pick it up.",True,True,True,True,[],[],[],[],"{'actions_ok': True, 'locations_ok': True, 'me...",/Users/jonathansalfity/Documents/dev/pyrobosim...,None,SUCCESS,8028,9,True,True,True


In [11]:
# If archetype is missing/unlabeled, infer from task id ranges in tasks50_roscon.yaml
def infer_archetype(task_id: str) -> str:
    m = re.match(r'task(\d+)_', str(task_id))
    if not m:
        return 'Unknown'
    n = int(m.group(1))
    if 1 <= n <= 10:
        return 'Sequential Hold'
    if 11 <= n <= 20:
        return 'Selector Search+Hold'
    if 21 <= n <= 30:
        return 'Pick+Place'
    if 31 <= n <= 40:
        return 'Selector Search+Place'
    if 41 <= n <= 50:
        return 'Selector Search+Return'
    return 'Unknown'

if 'archetype' not in df.columns:
    df['archetype_effective'] = df['id'].map(infer_archetype)
else:
    df['archetype_effective'] = df['archetype'].fillna('')
    mask = df['archetype_effective'].isin(['', 'unlabeled'])
    df.loc[mask, 'archetype_effective'] = df.loc[mask, 'id'].map(infer_archetype)

df[['id', 'archetype', 'archetype_effective']].head(10) if 'archetype' in df.columns else df[['id', 'archetype_effective']].head(10)

,id,archetype,archetype_effective
0,task01_hold_bread,unlabeled,Sequential Hold
1,task02_hold_snacks,unlabeled,Sequential Hold
2,task03_hold_soda,unlabeled,Sequential Hold
3,task04_hold_butter,unlabeled,Sequential Hold
4,task05_hold_waste_desk,unlabeled,Sequential Hold
5,task06_hold_waste_bin,unlabeled,Sequential Hold
6,task07_hold_bread_return_dining,unlabeled,Sequential Hold
7,task08_hold_soda_return_office,unlabeled,Sequential Hold
8,task09_hold_snacks_return_kitchen,unlabeled,Sequential Hold
9,task10_hold_butter_return_closet,unlabeled,Sequential Hold


In [12]:
# Keep evaluable rows
work = df.copy()
if 'missing_bt' in work.columns:
    work = work[~work['missing_bt'].fillna(False)]

metric_cols = ['valid', 'grounded', 'params_valid', 'struct_comp', 'success']
for c in metric_cols:
    if c in work.columns:
        work[c] = work[c].astype('boolean')

work.shape

(50, 21)

In [13]:
def rate(series: pd.Series):
    s = series.dropna()
    if len(s) == 0:
        return pd.NA
    return round(float(s.mean() * 100), 1)

def summarize_block(block: pd.DataFrame, name: str):
    return {
        'Archetype': name,
        'n': int(len(block)),
        'Valid': rate(block['valid']),
        'Grounded': rate(block['grounded']),
        'Params': rate(block['params_valid']),
        'StructComp': rate(block['struct_comp']),
        'TraceComp': '--',  # placeholder until trace metrics are implemented
        'Success': rate(block['success']),
    }

In [14]:
order = [
    'Sequential Hold',
    'Selector Search+Hold',
    'Pick+Place',
    'Selector Search+Place',
    'Selector Search+Return',
]

rows_out = [summarize_block(work, 'All tasks')]
for a in order:
    block = work[work['archetype_effective'] == a]
    rows_out.append(summarize_block(block, a))

summary_df = pd.DataFrame(rows_out)
summary_df

,Archetype,n,Valid,Grounded,Params,StructComp,TraceComp,Success
0,All tasks,50,100.0,100.0,100.0,100.0,--,88.0
1,Sequential Hold,10,100.0,100.0,100.0,100.0,--,100.0
2,Selector Search+Hold,10,100.0,100.0,100.0,100.0,--,90.0
3,Pick+Place,10,100.0,100.0,100.0,100.0,--,100.0
4,Selector Search+Place,10,100.0,100.0,100.0,100.0,--,90.0
5,Selector Search+Return,10,100.0,100.0,100.0,100.0,--,60.0


In [15]:
# Optional: execution status distribution
status_df = (
    work.groupby(['archetype_effective', 'exec_status'])
    .size()
    .unstack(fill_value=0)
    .reindex(order, fill_value=0)
)
status_df

exec_status,SUCCESS,TIMEOUT
archetype_effective,,
Sequential Hold,10,0
Selector Search+Hold,9,1
Pick+Place,10,0
Selector Search+Place,9,1
Selector Search+Return,6,4


In [16]:
# LaTeX helper lines for Table II (M-Core column values)
for _, r in summary_df.iterrows():
    print(f"{r['Archetype']} (n={int(r['n'])}): Valid={r['Valid']}, Grounded={r['Grounded']}, Params={r['Params']}, StructComp={r['StructComp']}, TraceComp={r['TraceComp']}, Success={r['Success']}")

All tasks (n=50): Valid=100.0, Grounded=100.0, Params=100.0, StructComp=100.0, TraceComp=--, Success=88.0
Sequential Hold (n=10): Valid=100.0, Grounded=100.0, Params=100.0, StructComp=100.0, TraceComp=--, Success=100.0
Selector Search+Hold (n=10): Valid=100.0, Grounded=100.0, Params=100.0, StructComp=100.0, TraceComp=--, Success=90.0
Pick+Place (n=10): Valid=100.0, Grounded=100.0, Params=100.0, StructComp=100.0, TraceComp=--, Success=100.0
Selector Search+Place (n=10): Valid=100.0, Grounded=100.0, Params=100.0, StructComp=100.0, TraceComp=--, Success=90.0
Selector Search+Return (n=10): Valid=100.0, Grounded=100.0, Params=100.0, StructComp=100.0, TraceComp=--, Success=60.0
